# Step 7 - Drag + rolling resistance
> Use that coast down data to determine vehicle drag + rolling resistance.

While coasting only drag and rolling resistance slow the car:

$m_{eff} \cdot \frac{dv}{dt} = -\frac{1}{2} \rho C_d A v^2 - C_{rr} m g$

which is a straight line of dv/dt against v²:

$\frac{dv}{dt} = \underbrace{-\frac{\frac{1}{2} \rho C_d A}{m_{eff}}}_{slope} v^2 \; \underbrace{- \frac{C_{rr} m g}{m_{eff}}}_{intercept}$

In [ ]:
import polars as pL
import numpy as np
import matplotlib.pyplot as plt

pd = pL.read_parquet("../software-data.parquet")
pd = pd.fill_null(strategy="forward")

rpm = pd["SME_TRQSPD_Speed"]
time = pd["Time"]

gear_ratio = 12/41
radius = .2                             #in meters

def meters_second(rpm):
    speed_mps = (rpm * gear_ratio * 2 * 3.14159 * radius) / 60  # Convert rpm to m/s
    return speed_mps

speed = meters_second(rpm)
moving = speed > 0.1

In [ ]:
def find_periods(condition):
    # turns a True/False mask into lists of start and end times
    # merges gaps under 0.1 s, drops periods 0.7 s or shorter
    total_periods = time.filter(condition)
    gaps = total_periods.diff()
    starts = total_periods.filter(gaps > .1)
    ends = total_periods.shift(1).filter(gaps > .1)

    starts = starts.to_list()
    ends = ends.to_list()
    total_periods = total_periods.to_list()
    starts = [total_periods[0]] + starts
    ends = ends + [total_periods[-1]]

    filtered_starts = []
    filtered_ends = []
    for s, e in zip(starts, ends):
        if e - s > 0.7:
            filtered_starts.append(s)
            filtered_ends.append(e)

    return filtered_starts, filtered_ends

Steps 4 - 6 recap: lap windows, coasting, and the clean coast segments.

In [ ]:
# lap windows from Step 4
lap_mask = (time >= 20) & (time <= 68)
lap_mask2 = (time >= 95) & (time <= 133)

brake = pd["ETC_STATUS_BRAKE_SENSE_VOLTAGE"]
pedal = pd["ETC_STATUS_PEDAL_TRAVEL"]

# Step 5 coasting: no throttle, no brake, still moving
no_throttle = pedal == 0
no_brake = brake <= 375
coasting_5 = no_throttle & no_brake & moving

# Step 6: coasting inside the laps, above 5 m/s, at least 1 s long
# speed cut first, then the 1 s check on what's left
in_laps = lap_mask | lap_mask2
fast_enough = speed >= 5
coast_6 = coasting_5 & in_laps & fast_enough

c6_starts, c6_ends = find_periods(coast_6)

clean_starts = []
clean_ends = []
for s, e in zip(c6_starts, c6_ends):
    if e - s >= 1:
        clean_starts.append(s)
        clean_ends.append(e)

# keep each coast as its own time + speed segment (Step 7 needs dv/dt inside each one)
coast_segments = []
for s, e in zip(clean_starts, clean_ends):
    seg_mask = (time >= s) & (time <= e) & fast_enough   # also drop the few dips under 5 m/s
    seg_time = time.filter(seg_mask)
    seg_speed = speed.filter(seg_mask)
    coast_segments.append((seg_time, seg_speed))

## dv/dt and v²
dv/dt uses **consecutive readings**, as the instructions say: change in speed / time between them. It's done inside each coast separately, and divides by the real time step since the segments have small holes.

In [ ]:
all_dvdt = []
all_v2 = []
for seg_time, seg_speed in coast_segments:
    dvdt = seg_speed.diff() / seg_time.diff()   # real dt, segments have small holes
    v2 = seg_speed ** 2
    keep = dvdt.is_not_null()                   # first row of each segment has no previous reading
    all_dvdt += dvdt.filter(keep).to_list()
    all_v2 += v2.filter(keep).to_list()

all_dvdt = pL.Series(all_dvdt)
all_v2 = pL.Series(all_v2)
print(f"points: {len(all_dvdt)}  (dv/dt mean {all_dvdt.mean():.3f} m/s^2)")

## Fit

In [ ]:
# straight line fit: dv/dt = slope * v^2 + intercept
(slope, intercept), cov = np.polyfit(all_v2.to_numpy(), all_dvdt.to_numpy(), 1, cov=True)
slope_err, intercept_err = np.sqrt(np.diag(cov))

air_density = 1.225     # kg/m^3
mass = 221.4            # kg
m_eff = 244.08          # kg
g = 9.81                # m/s^2

drag_term = -slope * m_eff              # .5 * p * Cd * A
cda = drag_term / (0.5 * air_density)   # Cd * A in m^2
crr = -intercept * m_eff / (mass * g)

cda_err = slope_err * m_eff / (0.5 * air_density)
crr_err = intercept_err * m_eff / (mass * g)

print(f"slope:     {slope:.6f} +/- {slope_err:.6f}")
print(f"intercept: {intercept:.4f} +/- {intercept_err:.4f}")
print(f".5*p*Cd*A: {drag_term:.4f}")
print(f"Cd*A:      {cda:.2f} +/- {cda_err:.2f} m^2")
print(f"Crr:       {crr:.4f} +/- {crr_err:.4f}")

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(all_v2, all_dvdt, s=8, alpha=0.3)   # lots of overlapping points, so make them see-through
plt.axhline(0, color='gray', linewidth=1)
plt.xlabel("v^2 (m^2/s^2)")
plt.ylabel("dv/dt (m/s^2)")
plt.title("Coastdown: dv/dt vs v^2")
plt.grid(True)

line_x = np.linspace(all_v2.min(), all_v2.max(), 100)
plt.plot(line_x, slope * line_x + intercept, color='red', linewidth=2, label="fit")

# each coast's average slowdown as a big dot, so the trend shows through the noise
for seg_time, seg_speed in coast_segments:
    avg_v2 = (seg_speed ** 2).mean()
    avg_dvdt = (seg_speed[-1] - seg_speed[0]) / (seg_time[-1] - seg_time[0])
    plt.scatter(avg_v2, avg_dvdt, s=80, color='black', zorder=3)
plt.scatter([], [], s=80, color='black', label="coast average")   # one legend entry for the dots
plt.legend()
plt.show()

Zoomed in so the fit line and coast averages are readable:

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(all_v2, all_dvdt, s=8, alpha=0.3)
plt.plot(line_x, slope * line_x + intercept, color='red', linewidth=2, label="fit")
for seg_time, seg_speed in coast_segments:
    avg_v2 = (seg_speed ** 2).mean()
    avg_dvdt = (seg_speed[-1] - seg_speed[0]) / (seg_time[-1] - seg_time[0])
    plt.scatter(avg_v2, avg_dvdt, s=80, color='black', zorder=3)
plt.scatter([], [], s=80, color='black', label="coast average")
plt.ylim(-1, 0.5)
plt.xlabel("v^2 (m^2/s^2)")
plt.ylabel("dv/dt (m/s^2)")
plt.title("Coastdown: dv/dt vs v^2 (zoomed)")
plt.legend()
plt.grid(True)
plt.show()

## Result

| | Value | 1 std. error range |
|---|---|---|
| Cd·A | 0.61 m² | about -0.13 to 1.35 m² |
| Crr | 0.030 | about 0.018 to 0.042 |

**Crr is roughly known (~0.02 - 0.04). Cd·A is not pinned down by this data:** the uncertainty in the slope is bigger than the slope itself, so the data can't rule out zero drag. 0.61 m² is the best-fit value, not a measurement.

**Why it's so noisy:** the speed sensor updates only every ~2nd row, so 58% of the dv/dt values are exactly 0 and the whole change lands in the next 0.012 s step. dv/dt spreads about ±8 m/s², while the real slowdown is only ~0.35 m/s². The scatter does average out (mean -0.349 m/s², matching each coast's overall slowdown), so the fit is centered on the right values - it just isn't precise.

**Why drag is hard to measure here:** the coasts only cover 5 - 10.7 m/s. With the best-fit values, drag is ~9 N at 5 m/s and ~43 N at 10.7 m/s, while rolling resistance is ~65 N throughout. Faster, longer coasts would be needed.

**Cross-check:** with a 0.5 s window (`.diff(42)`) instead, Cd·A = 1.19 m² and Crr = 0.022. Both are inside the ranges above, but Cd·A doubles - another sign drag isn't well determined.

**Plausibility:** Cd·A ~0.6 m² is a reasonable size for a car without wings. Crr ~0.03 is about 2x tires alone (~0.01 - 0.015), but this "rolling resistance" includes everything that doesn't grow with v²: chain and bearing friction, motor drag, and any slope.

**Slope caveat:** ~29 s of the clean coasts head east and only ~9 s west, so a hill wouldn't fully cancel. The west coast slows about the same as the east ones at similar speed, which suggests the slope is small, but GPS altitude is too coarse to prove it.